# Credit Risk Scoring Model

**Business Question:** Given an applicant's financial and demographic profile, what is the probability they will default on their loan?

**Dataset:** German Credit Dataset (1,000 applicants, 20 features, binary default label)  
**Source:** OpenML Dataset #31

---

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, roc_auc_score, roc_curve,
    confusion_matrix, ConfusionMatrixDisplay
)
from sklearn.pipeline import Pipeline

import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

## 1. Load Data

In [ ]:
# Load German Credit dataset from OpenML
dataset = fetch_openml(data_id=31, as_frame=True, parser='auto')

df = dataset.frame
print(f"Shape: {df.shape}")
df.head()

In [ ]:
# Target: 'class' — 'good' (no default) or 'bad' (default)
print(df['class'].value_counts())
print(f"\nDefault rate: {(df['class'] == 'bad').mean():.1%}")

In [ ]:
df.info()

## 2. Exploratory Data Analysis

Before building a model, we need to understand the data — what signals predict default, and where the risk concentrates.

In [ ]:
# Class balance
fig, ax = plt.subplots(figsize=(6, 4))
df['class'].value_counts().plot(kind='bar', ax=ax, color=['steelblue', 'tomato'], edgecolor='white')
ax.set_title('Loan Outcome Distribution', fontsize=14)
ax.set_xlabel('Outcome')
ax.set_ylabel('Count')
ax.set_xticklabels(['Good (No Default)', 'Bad (Default)'], rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of credit amount by outcome
fig, ax = plt.subplots(figsize=(8, 4))
for label, color in [('good', 'steelblue'), ('bad', 'tomato')]:
    df[df['class'] == label]['credit_amount'].plot(
        kind='hist', bins=30, alpha=0.6, ax=ax, color=color, label=label
    )
ax.set_title('Credit Amount by Loan Outcome', fontsize=14)
ax.set_xlabel('Credit Amount')
ax.legend(title='Outcome')
plt.tight_layout()
plt.show()

In [ ]:
# Default rate by checking account status (one of the strongest predictors in this dataset)
default_by_checking = df.groupby('checking_status')['class'].apply(
    lambda x: (x == 'bad').mean()
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
default_by_checking.plot(kind='bar', ax=ax, color='tomato', edgecolor='white')
ax.set_title('Default Rate by Checking Account Status', fontsize=14)
ax.set_ylabel('Default Rate')
ax.set_xlabel('Checking Account Status')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 3. Feature Engineering

In [ ]:
# Separate features and target
X = df.drop(columns=['class'])
y = (df['class'] == 'bad').astype(int)  # 1 = default, 0 = no default

# Encode categorical features
X_encoded = pd.get_dummies(X, drop_first=True)

print(f"Features after encoding: {X_encoded.shape[1]}")
X_encoded.head()

In [ ]:
# Train/test split — stratified to preserve default rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train size: {X_train.shape[0]} | Test size: {X_test.shape[0]}")
print(f"Default rate — Train: {y_train.mean():.1%} | Test: {y_test.mean():.1%}")

## 4. Baseline Model: Logistic Regression

Logistic regression is the industry standard for credit scoring. It's interpretable, auditable, and its output is a probability — exactly what a loan officer needs.

In [ ]:
# Build pipeline: scale features + fit logistic regression
lr_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

lr_pipeline.fit(X_train, y_train)

# Evaluate
y_pred_lr = lr_pipeline.predict(X_test)
y_proba_lr = lr_pipeline.predict_proba(X_test)[:, 1]

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr, target_names=['No Default', 'Default']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_lr):.3f}")

## 5. Improved Model: XGBoost

XGBoost captures nonlinear interactions between features that logistic regression misses. We compare both and let the business context determine which to deploy.

In [ ]:
from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    use_label_encoder=False,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("=== XGBoost ===")
print(classification_report(y_test, y_pred_xgb, target_names=['No Default', 'Default']))
print(f"ROC-AUC: {roc_auc_score(y_test, y_proba_xgb):.3f}")

## 6. Model Comparison

In [ ]:
# ROC curves
fig, ax = plt.subplots(figsize=(7, 5))

for name, proba in [('Logistic Regression', y_proba_lr), ('XGBoost', y_proba_xgb)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    auc = roc_auc_score(y_test, proba)
    ax.plot(fpr, tpr, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curve Comparison', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

## 7. Business Interpretation

Accuracy metrics alone don't tell the lender what to do. This section answers the real question: **at a given approval threshold, what does the portfolio look like?**

In [ ]:
# Approval threshold analysis
# Approve applicants with predicted default probability BELOW the threshold

thresholds = np.arange(0.2, 0.8, 0.05)
results = []

for threshold in thresholds:
    approved = y_proba_xgb < threshold
    if approved.sum() == 0:
        continue
    actual_defaults = y_test[approved].mean()
    approval_rate = approved.mean()
    results.append({
        'Threshold': f'{threshold:.0%}',
        'Approval Rate': f'{approval_rate:.0%}',
        'Default Rate Among Approved': f'{actual_defaults:.0%}',
        'Applicants Approved': int(approved.sum())
    })

results_df = pd.DataFrame(results)
print("Approval Threshold Analysis (XGBoost)")
print(results_df.to_string(index=False))

---

## Summary & Recommendation

*(To be completed after model results are finalized)*

**Key findings:**
- The XGBoost model achieves a ROC-AUC of `[X]` vs `[X]` for logistic regression
- At a `[X]%` approval threshold, the lender can approve `[X]%` of applicants with an expected default rate of `[X]%`
- The strongest predictors of default are: `[to be filled]`

**Recommendation:** Use the XGBoost model with a `[X]%` probability threshold for production. This gives the lender a calibrated, auditable decision tool that reduces default exposure while maintaining a competitive approval rate.